<div style="text-align: center;">
 <img width=700px heigth=20px src="./IMG/GPT_architect.png" alt="Example image">
 </div>

In [30]:
import torch
import torch.nn as nn

In [31]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.epsilon=1e-5
    self.shift=nn.Parameter(torch.zeros(emb_dim))
    self.scale=nn.Parameter(torch.ones(emb_dim))
  def forward(self,inp):
    mean = inp.mean(dim=-1, keepdim=True) 
    var = inp.var(dim=-1, keepdim=True)
    input_Norm=(inp-mean)/torch.sqrt(var+self.epsilon)
    return input_Norm*self.scale+self.shift

In [32]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,dropout,qkv_bias=False):
    super().__init__()
    
    self.d_in=d_in
    self.d_out=d_out
    self.num_head=num_heads
    if(num_heads==0):
      num_heads+=0.000000001
    self.head_dim=d_out//num_heads
    
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias) #layer not the actual wt
    self.w_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.context_length=context_length
    self.dropout=nn.Dropout(dropout)
    
    self.qkv_bias=qkv_bias
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))#register_buffer makes PyTorch officially aware of this tensor:here var is mask

  def forward(self,x):
    batch_size,num_token,d_in=x.shape
    
    keys=self.w_key(x)
    querys=self.w_query(x)
    values=self.w_value(x)
    
    #split using view    -- converting (batch_size,num_token,d_in) =-> (batch_size,num_token,num_head,head_dim)
    keys=keys.view(batch_size,num_token,self.num_head,self.head_dim)
    querys=querys.view(batch_size,num_token,self.num_head,self.head_dim)
    values=values.view(batch_size,num_token,self.num_head,self.head_dim)
    
    #taking the transpose and grouping according to the head  converting---> (batch_size,num_token,num_head,head_dim)=-> (batch_size,num_head,num_token,head_dim)
    keys=keys.transpose(1,2)
    querys=querys.transpose(1,2)
    values=values.transpose(1,2)
    
    atten_scores=querys@keys.transpose(2,3)
    
    mask_bool=self.mask.bool()[:num_token,:num_token]
    atten_scores.masked_fill(mask_bool,-torch.inf)
    
    attn_weigth=torch.softmax(atten_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weigth=self.dropout(attn_weigth)
    
    context_vec=(attn_weigth@values).transpose(1,2)
    
    context_vec=context_vec.contiguous().view(batch_size,num_token,self.d_out)
    context_vec = self.out_proj(context_vec)
    
    return context_vec    

In [33]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

In [34]:
class FeedForward(nn.Module):
  def __init__(self,GPT_2_config):
    super().__init__()
    self.layer=nn.Sequential(
      nn.Linear(GPT_2_config["emb_dim"],4 * GPT_2_config["emb_dim"]), #expansion
      GELU()  ,               #activation
      nn.Linear(4 * GPT_2_config["emb_dim"], GPT_2_config["emb_dim"]) #contraction
    )
  def forward(self,x):
    return self.layer(x)  

In [35]:
cfg=GPT_2_config={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":768,
  "n_head":12,
  "n_layer":12,
  "dropout":0.1,
  "qkv_bias":False  
}

In [36]:
class Transfomers(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layernorm=LayerNorm(cfg["emb_dim"])
    self.mutihead_atten=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_len"],
      num_heads=cfg["n_head"],
      dropout=cfg["dropout"],
      qkv_bias=False)
    self.dropout=nn.Dropout(cfg["dropout"])
    self.feed_forward=FeedForward(cfg)
    
  def forward(self,x):
    shortcut=x
    
    x=self.layernorm.forward(x)
    x=self.mutihead_atten.forward(x)
    x=self.dropout(x)
    x=x+shortcut
    
    shortcut=x
    
    x=self.layernorm(x)
    x=self.feed_forward(x)
    x=self.dropout(x)
    x=x+shortcut
    
    return x

In [37]:
class GPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    
    self.token_emb=nn.Embedding(cfg["vocab"],cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_len"],cfg["emb_dim"])
    self.dropout=nn.Dropout(cfg["dropout"])
    self.tranformer_block= nn.Sequential(
            *[Transfomers(cfg) for _ in range(cfg["n_layer"])]
        )
    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab"])
    
    
  def forward(self,ip_batch):
      batch,seq_len=ip_batch.shape
      
      token_embd=self.token_emb(ip_batch)
      pos_embd = self.pos_emb(torch.arange(seq_len, device=ip_batch.device))
      
      x=token_embd+pos_embd
      
      x=self.dropout(x)
      
      x=self.tranformer_block(x)
      
      x=self.final_norm(x)
      
      logit=self.out_head(x)
      
      return logit
       

In [38]:
from GPT_Model import text_to_token_id,tokenizer

In [39]:
batch_text=["After effort move you","Every day holds very"]
batch_token_id=text_to_token_id(batch_text,tokenizer)
print(batch_token_id)

tensor([[3260, 3626, 1445,  345],
        [6109, 1110, 6622,  845]])


In [40]:
torch.manual_seed(123)

model = GPT(cfg)

out = model(batch_token_id)

print("INPUT BATCH: ", batch_token_id.shape)
print("OUTPUT BATCH SHAPE: ", out.shape)


INPUT BATCH:  torch.Size([2, 4])
OUTPUT BATCH SHAPE:  torch.Size([2, 4, 50257])


In [41]:
total_params = sum(p.numel() for p in model.parameters())

print(total_params)

163041361


In [42]:
print("Input Embedding Layer Shape: ", model.token_emb.weight.shape)
print("Output Layer Shape: ", model.out_head.weight.shape)

Input Embedding Layer Shape:  torch.Size([50257, 768])
Output Layer Shape:  torch.Size([50257, 768])


#### ``weight typing`` is used that mean in GPT output embeding and input embeding are same
## why we get ```163041361``` parameter instead of    ```124393728``` in GPT-2 ``?``
 because in calucation we same embeding in output and input but calcualation count twice so we have to substract it

In [43]:
total_params_actual=total_params - sum(p.numel() for p in model.out_head.parameters())
total_params_actual

124393728

## Memory of model 

In [44]:
#convert to bit assume 32 bit taken by one param so
total_size=total_params_actual*4 #in bit
total_size_in_mb=total_size/(1024*1024) #convert to mb (bit->kb->mb)
print(f"total size of GPT-2 model weight is {total_size_in_mb:2f}mb")

total size of GPT-2 model weight is 474.524414mb
